# 🔍 Lab 05: RAG 問答系統 (Retrieval-Augmented Generation)

**使用 LangChain + FAISS 構建中文 RAG**

---

### 🎯 學習目標
- 文檔載入與切分
- 使用 sentence-transformers 做中文嵌入
- FAISS 向量檢索
- 互動問答 + 評估檢索質量

### 📊 數據集
8 個範本中文文檔（台積電/NVIDIA/AI/區塊鏈/量子計算/張忠謀/黃仁勳/Elon Musk）

### 🛠 技術棧
LangChain, FAISS, sentence-transformers, HuggingFace

### ⏱ 預計時間
- GPU: 不需要
- 嵌入模型首次下載: ~30 秒
- 運行時間: ~2 分鐘

### 💡 兩種模式
- **本機模式 (無需 API key)**：使用本地嵌入 + 簡單提取式回答
- **LLM 模式 (需 OpenAI key)**：完整生成式問答

---

## 1. 安裝

In [ ]:
!pip install -q langchain langchain-community langchain-huggingface faiss-cpu sentence-transformers
import os
print('✅ 安裝完成')

## 2. 載入文檔

In [ ]:
from pathlib import Path
from langchain_community.document_loaders import DirectoryLoader, TextLoader

DOCS_DIR = '/content/docs' if os.path.exists('/content') else './docs'
if not os.path.exists(f'{DOCS_DIR}/01_tsmc.txt'):
    os.makedirs(DOCS_DIR, exist_ok=True)
    for f in ['01_tsmc', '02_nvidia', '03_ai', '04_blockchain', '05_quantum', '06_morris_chang', '07_jensen_huang', '08_elon_musk']:
        !wget -q https://raw.githubusercontent.com/chanxaviersy/nlp-ml-knowledge-base/main/docs/08-labs/05-rag-qa-system/data/documents/{f}.txt -O {DOCS_DIR}/{f}.txt

loader = DirectoryLoader(DOCS_DIR, glob='**/*.txt', loader_cls=TextLoader,
                          loader_kwargs={'encoding': 'utf-8'})
docs = loader.load()
print(f'✅ 載入 {len(docs)} 個文檔')
for d in docs:
    print(f'   - {Path(d.metadata["source"]).name}: {len(d.page_content)} 字')

## 3. 文檔切分

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=300, chunk_overlap=50,
    separators=['\n\n', '\n', '。', '.', ' ', ''],
)
chunks = splitter.split_documents(docs)
print(f'✂️  切分為 {len(chunks)} 個塊')

## 4. 構建向量庫

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

print('🔧 載入嵌入模型 (首次約 30 秒)...')
embeddings = HuggingFaceEmbeddings(
    model_name='sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2',
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True},
)

print('📦 構建向量庫...')
vectorstore = FAISS.from_documents(chunks, embeddings)
print('✅ 向量庫構建完成')

## 5. 評估檢索

In [ ]:
test_queries = [
    ('台積電的創辦人是誰？', ['張忠謀', '1987']),
    ('NVIDIA 是誰創辦的？', ['黃仁勳', '1993']),
    ('什麼是區塊鏈？', ['去中心化', '帳本']),
    ('量子計算的單位是什麼？', ['量子位元', 'qubit']),
    ('Elon Musk 創辦了哪些公司？', ['Tesla', 'SpaceX']),
    ('黃仁勳的學歷是？', ['Stanford', '碩士']),
]

correct = 0
for q, keywords in test_queries:
    results = vectorstore.similarity_search(q, k=3)
    retrieved = ' '.join([d.page_content for d in results])
    hit = any(kw in retrieved for kw in keywords)
    correct += int(hit)
    icon = '✅' if hit else '❌'
    print(f'{icon} Q: {q}')

print(f'\n📊 命中率: {correct}/{len(test_queries)} = {correct/len(test_queries):.0%}')

## 6. 互動問答（提取式）

In [ ]:
def ask(query, k=3):
    docs = vectorstore.similarity_search(query, k=k)
    print(f'\n❓ 問題: {query}')
    print(f'\n📚 參考來源 (top-{k}):')
    for i, d in enumerate(docs, 1):
        src = Path(d.metadata.get('source', '')).name
        print(f'  [{i}] {src}: {d.page_content[:120]}...')
    answer = docs[0].page_content[:200].replace('\n', ' ')
    print(f'\n💡 簡短回答: {answer}...')

for q in [
    '台積電的創辦人是誰？',
    'NVIDIA 做什麼的？',
    '區塊鏈是什麼？',
    '量子計算能用在哪裡？',
]:
    ask(q)

## 7. (可選) 完整 LLM 模式

如果設置了 `OPENAI_API_KEY` 環境變量，可運行完整生成式 RAG。

In [ ]:
import os

# 取消下面註釋以使用 LLM（需 OpenAI API key）
# os.environ['OPENAI_API_KEY'] = 'sk-...'

if os.getenv('OPENAI_API_KEY'):
    from langchain_openai import ChatOpenAI
    from langchain.chains import RetrievalQA
    from langchain.prompts import PromptTemplate

    template = """使用以下上下文回答問題。如果不知道就說不知道，不要編造。

上下文:
    {context}

問題: {question}
回答:"""
    prompt = PromptTemplate(template=template, input_variables=['context', 'question'])

    llm = ChatOpenAI(model='gpt-3.5-turbo', temperature=0)
    qa = RetrievalQA.from_chain_type(
        llm=llm, chain_type='stuff',
        retriever=vectorstore.as_retriever(search_kwargs={'k': 3}),
        chain_type_kwargs={'prompt': prompt},
        return_source_documents=True,
    )
    result = qa({'query': '台積電的創辦人是誰？'})
    print(result['result'])
else:
    print('⏭️  跳過 LLM 模式（未設置 OPENAI_API_KEY）')

## 🎉 恭喜完成 Lab 05！

**你學到了**：
- ✅ 文檔載入與切分
- ✅ 中文嵌入模型 (sentence-transformers)
- ✅ FAISS 向量檢索
- ✅ 檢索質量評估 (Hit Rate)
- ✅ RAG 完整流程

### 🚀 進階方向

- **Cross-Encoder Re-ranking**：用更精確模型重排檢索結果
- **Hybrid Search**：BM25 + 嵌入
- **HyDE**：生成假設文檔做檢索
- **部署**：FastAPI + Streamlit
- **GraphRAG**：知識圖譜增強

### 📚 延伸閱讀

- [LangChain RAG 教程](https://python.langchain.com/docs/use_cases/question_answering/)
- [FAISS 文檔](https://faiss.ai/)
- [BGE 嵌入模型](https://huggingface.co/BAAI/bge-m3)
- [RAGAS 評估框架](https://docs.ragas.io/)